# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sidd13789/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1

The research paper reports that content freshness is associated with improved search performance.

### Methodology Question

How was the target label created, and over what time period was the improvement measured? A time-aware validation design would provide stronger evidence that the observed relationship generalizes to future data.

---

## Finding 2

The paper reports that combining multiple content signals improves prediction quality.

### Methodology Question

Was validation performed on completely unseen clients or only on randomly selected samples? Grouped validation by client would better evaluate whether the model generalizes across different websites.italicized text

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## Honest Validation

In Week 5, I evaluated the model using a random 80/20 train-test split.

For this audit, I used a grouped split based on `client_id`. This prevents content from the same client appearing in both training and testing data, providing a more realistic estimate of model performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [9]:
df = pd.read_csv("/content/content_refresh_anonymized.csv")

In [10]:
X = df.drop(columns=[
    "trend_direction",
    "trend_pct"          # remove leakage
])

y = df["trend_direction"]

groups = df["client_id"]

In [11]:
for col in X.columns:
    if X[col].dtype == "object":
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

y = LabelEncoder().fit_transform(y)

In [12]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

In [13]:
model = RandomForestClassifier(
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

group_accuracy = accuracy_score(y_test, pred)

print(group_accuracy)

0.7290280707447672


In [14]:
comparison = pd.DataFrame({
    "Validation Strategy":[
        "Random Split (Week 5)",
        "Grouped Split (Week 6)"
    ],
    "Accuracy":[
        0.9995,
        group_accuracy
    ]
})

comparison

,Validation Strategy,Accuracy
0,Random Split (Week 5),0.999500
1,Grouped Split (Week 6),0.729028


## Leakage Audit

During this audit I reviewed the final feature set for potential target leakage.

Findings:

- `trend_pct` was strongly related to the target (`trend_direction`) and could leak information.
- This feature was removed before retraining the model.
- The grouped validation also prevented records from the same client appearing in both training and testing data.
- No direct target column was included in the final feature set.

These steps provide a more realistic estimate of model performance.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Original Claim

"The model predicts trend direction accurately."

---

## Revised Claim

The model showed strong measured performance on the evaluated dataset under the selected validation strategy.

These results are observational and should be interpreted as decision-support rather than evidence that the model will generalize to every future dataset.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.